# Puckworks — CPU quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trbrewer/puckworks/blob/main/notebooks/puckworks_quickstart_colab.ipynb)

**Puckworks turns espresso-process research into small, auditable software components, tests them
against real measurements, and is careful to say what the data can — and cannot — support.**

It is a *registry* of models, not one giant "mega-model". Each published model becomes a
component that carries where it came from, what it assumes, the range it is valid over, and the
validation gates it must pass.

This notebook is for a **first look**. It:

- runs on a normal Colab **CPU** — no GPU, no account secrets, no private data;
- installs the **latest public release** ([`v0.2.0`](https://github.com/trbrewer/puckworks/releases/tag/v0.2.0)) and **verifies its
  SHA-256 before installing** — not unreleased development code;
- takes a few minutes end to end.

> This notebook *uses* the installed `puckworks` package. It never copies the package's own code
> into these cells — what you run here is exactly what a user gets from the release.

Run the cells top to bottom (Runtime ▸ Run all).

## 1. Install the exact public release (with hash verification)

By default this **downloads** the canonical release wheel `puckworks-0.2.0-py3-none-any.whl` from the
[v0.2.0 GitHub Release](https://github.com/trbrewer/puckworks/releases/tag/v0.2.0), **verifies its SHA-256 against the recorded canonical
hash**, and installs the verified local file. On a hash mismatch it refuses to install.

Automated tests set `PUCKWORKS_WHEEL` to a locally built (development) wheel for a hermetic,
network-free run; that path installs the local file directly and skips the release-hash check
because a development wheel has a different hash. On Colab you can ignore that — the default
verified-release path is used. puckworks is **not on PyPI**.

In [ ]:
import hashlib
import os
import subprocess
import sys
import urllib.request

# Canonical public-release facts — kept in sync with docs/status/public_release.json
# (tests/test_readme.py enforces equality). puckworks is NOT on PyPI.
WHEEL_FILENAME = 'puckworks-0.2.0-py3-none-any.whl'
WHEEL_URL = 'https://github.com/trbrewer/puckworks/releases/download/v0.2.0/puckworks-0.2.0-py3-none-any.whl'
WHEEL_SHA256 = '0a54f8782e4173132308af24ce0c88bb48a2d334071382c07c0ff5ecc9082df1'


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


override = os.environ.get("PUCKWORKS_WHEEL", "").strip()
if override:
    # Hermetic / CI mode: install a locally built development wheel, no network, no hash gate.
    print("Hermetic mode — installing local wheel:", override)
    target = override
else:
    print("Expected wheel SHA-256:", WHEEL_SHA256)
    target = os.path.join("/tmp", WHEEL_FILENAME)
    urllib.request.urlretrieve(WHEEL_URL, target)
    actual = _sha256(target)
    print("Verified wheel SHA-256:", actual)
    if actual != WHEEL_SHA256:
        raise SystemExit("Wheel SHA-256 mismatch — refusing to install (expected "
                         f"{WHEEL_SHA256}, got {actual}).")
    print("Hash OK — installing the verified local file.")

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", target], check=True)
print("Install complete.")

## 2. Import and environment check

Confirm the released package imported, and see exactly which build and interpreter you are on.

In [ ]:
import platform

import puckworks

print("puckworks version :", puckworks.__version__)
print("Python version    :", platform.python_version())
print("Running on        :", platform.system(), platform.machine())
print("Imported from     :", puckworks.__file__)

## 3. Explore the registry

Every component records its **source (provenance)**, its **stage/role** in the espresso process,
and its **validity range and assumptions**. Nothing is anonymous.

In [ ]:
puckworks.load_builtin_components()
components = puckworks.components()
print(f"{len(components)} components registered\n")

for c in components[:5]:
    print(f"• {c.name}")
    print(f"    stage / role : {c.stage}  ·  kind={c.kind}")
    print(f"    source       : {c.paper}")
    print(f"    provenance   : {c.provenance_class}  ·  evidence={c.evidence_strength}")
    print(f"    valid range  : {c.valid_range}")
    print()

## 4. Run the validation gates

A gate compares a component against real data. Each gate ends in a **status**. The two you will
see here are:

- **PASS** — the component met the gate's criterion.
- **ACKNOWLEDGED_EXCEPTION** — a *known, documented* deviation that the maintainers have accepted
  and recorded. **It is deliberately not called a PASS.** Reporting it separately is the whole
  point: the registry does not launder a known limitation into a success.

In [ ]:
suite = puckworks.evaluate_all_gates()

counts = {str(getattr(k, "value", k)): int(v) for k, v in suite.counts_by_status.items()}
print(suite.summary_text())
print("\nBreakdown by status:")
for status, n in sorted(counts.items()):
    if n:
        print(f"    {status:24} {n}")

print("\nSuite passed under the documented gate policy (no genuine failures):", bool(suite.passed))

## 5. Visual summary

A simple, accessible chart of the gate outcomes. Every bar is **labelled in words** with its
status and count — the meaning never depends on colour alone. If plotting is unavailable the same
information prints as text.

In [ ]:
labels = ["PASS", "ACKNOWLEDGED_EXCEPTION", "FAIL", "ERROR", "SKIP"]
pairs = [(lbl, counts.get(lbl, 0)) for lbl in labels if counts.get(lbl, 0)]

try:
    import matplotlib.pyplot as plt
    _have_mpl = True
except ImportError as exc:
    print("(matplotlib unavailable:", exc, "- text summary instead)")
    _have_mpl = False

if _have_mpl:
    fig, ax = plt.subplots(figsize=(7, 2.4))
    ys = range(len(pairs))
    ax.barh(list(ys), [v for _, v in pairs], color="#0e7490")
    ax.set_yticks(list(ys))
    ax.set_yticklabels([lbl for lbl, _ in pairs])
    ax.invert_yaxis()
    ax.set_xlabel("number of gates")
    ax.set_title("Validation gate outcomes")
    for i, (lbl, v) in enumerate(pairs):
        ax.text(v, i, f"  {lbl}: {v}", va="center", fontsize=9)
    ax.margins(x=0.25)
    plt.tight_layout()
    plt.show()
else:
    for lbl, v in pairs:
        print(f"    {lbl:24} {'#' * v}  {v}")

## 6. Interpret responsibly

Puckworks labels *where every number comes from*. When you read a result, keep the origin in mind:

| label | means |
|---|---|
| **measured** | it came from an instrument / dataset |
| **derived** | computed from measured quantities by a defined relation |
| **fitted** | a parameter tuned so the model matches data (agreement here is *not* independent proof) |
| **predicted** | the model's output for conditions, ahead of a measurement |
| **simulated** | produced by a numerical solver |
| **unsupported** | the release cannot establish this — do not treat it as a result |

What this release ([`v0.2.0`](https://github.com/trbrewer/puckworks/releases/tag/v0.2.0)) **is**: an inspectable registry of components with
provenance and gate status. It makes assumptions, evidence, validity ranges, and model behavior
comparable.

What it is **not**: it does **not** diagnose your espresso shot, predict flavour, or optimise a
recipe. Most extraction "agreements" in the literature are *post-fit reconstruction*, not
independent validation — and Puckworks never upgrades that label for you.

## 7. Where to go next

- **README / project homepage** — https://github.com/trbrewer/puckworks
- **Explore the evidence** — [`docs/public/README.md`](https://github.com/trbrewer/puckworks/blob/main/docs/public/README.md)
- **Public API + stability policy** — [`docs/API.md`](https://github.com/trbrewer/puckworks/blob/main/docs/API.md)
- **Current project status** — [`docs/planning/STATE_OF_TRUTH.md`](https://github.com/trbrewer/puckworks/blob/main/docs/planning/STATE_OF_TRUTH.md)
- **Advanced GPU / lattice-Boltzmann notebook** — [`notebooks/espresso_lb_colab.ipynb`](https://github.com/trbrewer/puckworks/blob/main/notebooks/espresso_lb_colab.ipynb)
- **Contribute a model or dataset** — [`CONTRIBUTING.md`](https://github.com/trbrewer/puckworks/blob/main/CONTRIBUTING.md)
- **Release provenance** — https://github.com/trbrewer/puckworks/releases/tag/v0.2.0

In [ ]:
print("QUICKSTART_COMPLETE:", len(puckworks.components()), "components,",
      counts.get("PASS", 0), "PASS +",
      counts.get("ACKNOWLEDGED_EXCEPTION", 0), "ACKNOWLEDGED_EXCEPTION")